In [1]:
!which python3
!python3 --version

/opt/anaconda3/envs/xlsx/bin/python3
Python 3.13.13


Staff Schedule Upload

In [8]:
# TODO: open file picker to select input csv file (or xlsx file, tbd)

# input_file = Path("new_sched/input_files/2026B_NA_Staff_Schedule.csv")
# output_file = Path("new_sched/output_files/2026B_NA_Staff_Schedule.sql")

# print(f" Input file: {input_file}")
# print(f"Output file: {output_file}")

#!/usr/bin/env /opt/anaconda3/bin/python3

"""Parse staff schedule CSV files and emit SQL insert statements.

This is a Python port of updated PHP code
(which is a standalone CLI port of the original PHP uploader logic).

Desc: Parses CSV file with expected format for night staff scheduling data.
Outputs: Only generates SQL text and does not execute the inserts.

TODOs:
- Ignore obvious blank content lines and header.
- Check for sequential dates with no gaps.
- Warn unknown initials
- Get alias mapping from db?
- Add a confirm after parse to do the actual inserts and/or write to sql file
- Make enough validation to allow someone like Gloria to use
- Create cron job that sends out reminder if db is about out of dates.
- Could be easy to select the wrong radio type, so grouping initials mapping by type and only looking at that grouping.

- For NA, should it have a whitelist of acceptable types?
- `csv` in Python is a standard-library module, not an abstract class, so there are no abstract methods to implement.
- If the intent is OOP extensibility, define a project-local abstract parser interface (for example: `parse_lines`, `validate_row`, `to_sql`) and implement it for `oa`, `na`, `sa`, and `eeoc`.
- Current notebook logic already provides concrete behavior through `convert_oa`, `convert_na`, `convert_sa`, and `convert_eeoc`.
"""

In [2]:
from __future__ import annotations

import argparse
import csv
import sys
from datetime import datetime
from pathlib import Path
from typing import Iterable

In [3]:
INITIALS: dict[str, str] = {
    "CJ": "cjordan",
    "TS": "tstickel",
    "JA": "jaycock",
    "CW": "cwilburn",
    "JRK": "julierk",
    "HH": "hhershley",
    "JP": "jpelletier",
    "TR": "tridenour",
    "AH": "ahatakeyama",
    "AR": "arostopchina",
    "RM": "rmorris",
    "LF": "lfuhrman",
    "MW": "mwahl",
    "MP": "mpiper",
    "SJ": "sjoseph",
    "NJ": "njordan",
    "JLP": "jlapinta",
    "TC": "tconnors",
    "DO": "dorr",
    "KWB": "kbrennon",
    "AD": "adeverse",
    "LS": "lsato",
    "EL": "elevine",
    "SG": "sguerpo",
    "MM": "mmedeiros",
    "CB": "cbishop",
    "KK": "kkameoka-vincent",
}

INITIALS_SA: dict[str, str] = {
    "PG": "pgomez",
    "CA": "calvarez",
    "RC": "randyc",
    "GD": "gdoppmann",
    "JL": "jlyke",
    "JW": "jwalawender",
    "SY": "syeh",
    "ML": "mlundquist",
    "RM": "rmcgurk",
    "LA": "lalcorn",
    "KM": "kmatthews",
    "PK": "pkrishnamoorthy",
}

In [4]:
# print(f'Night Staff: {INITIALS.keys()}')
# print(f'SA Staff: {INITIALS_SA.keys()}')

In [20]:
# command line argument parsing

import argparse
from pathlib import Path

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Convert a staff schedule CSV into SQL insert statements."
    )
    parser.add_argument(
        "--type",
        required=True,
        choices=["oa", "na", "sa", "eeoc"],
        help="Schedule format to parse.",
    )
    parser.add_argument(
        "--input",
        required=True,
        type=Path,
        help="Input CSV path.",
    )
    parser.add_argument(
        "--output",
        type=Path,
        help="Optional output SQL file path. Default: stdout.",
    )
    parser.add_argument(
        "--verbose",
        action="store_true",
        help="Emit parse progress to stderr.",
    )
    return parser.parse_args()


In [ ]:
def read_lines(path: Path) -> list[str]:
    return path.read_text(encoding="utf-8", errors="replace").splitlines()


def parse_csv_line(line: str) -> list[str]:
    row = next(csv.reader([line]))
    return [item.strip() for item in row]


def parse_date(value: str) -> str:
    text = value.strip()
    formats = (
        "%Y-%m-%d",
        "%m/%d/%Y",
        "%m/%d/%y",
        "%b %d %Y",
        "%b %d, %Y",
        "%B %d %Y",
        "%B %d, %Y",
    )
    for fmt in formats:
        try:
            return datetime.strptime(text, fmt).strftime("%Y-%m-%d")
        except ValueError:
            continue

    # Fall back to datetime parsing for ISO-like variants.
    try:
        return datetime.fromisoformat(text).strftime("%Y-%m-%d")
    except ValueError as exc:
        raise ValueError(f"Unable to parse date: {value!r}") from exc


def sql_insert(date: str, telnr: str | int, alias: str, typ: str) -> str:
    return (
        "insert into nightStaff set "
        f"Date='{date}', "
        f"TelNr='{telnr}', "
        f"Alias='{alias}', "
        f"Type='{typ}';"
    )


def emit(lines: Iterable[str], output: Path | None) -> None:
    text = "\n".join(lines)
    if output is None:
        print(text)
        return
    output.write_text(text + "\n", encoding="utf-8")


def convert_oa(lines: list[str], verbose: bool) -> list[str]:
    skip = {
        "",
        "X",
        "x",
        "L",
        "T",
        "H",
        "OM",
        "HQ",
        "PD",
        "SD",
        "CDP",
        "CPR",
        "ELP",
        "KSM",
        "PR",
        "First Aid",
        "SMOWG",
        "TelSched",
    }
    for i in range(1, 100):
        skip.add(f"O{i}")
        skip.add(f"o{i}")

    code = {
        "K1": "oa",
        "K2": "oa",
        "R1": "oar",
        "R2": "oar",
        "K1O": "oao",
        "K2O": "oao",
        "R1O": "oaro",
        "R2O": "oaro",
        "K1T": "oat",
        "K2T": "oat",
        "R1T": "oart",
        "R2T": "oart",
    }

    header: list[str] = []
    out: list[str] = []

    for line in lines:
        line = line.strip()
        if not line:
            continue
        if verbose:
            print(f"OA line: {line}", file=sys.stderr, flush=True)
        split = parse_csv_line(line)

        if "Date,DOW" in line:
            header = [h for h in split if h]
            continue

        date = ""
        for key, hdr in enumerate(header):
            if key >= len(split):
                continue
            cell = split[key]
            if hdr == "Date" and not date:
                date = parse_date(cell)
            elif hdr in INITIALS and cell not in skip:
                if "1" in cell:
                    telnr: str | int = 1
                elif "2" in cell:
                    telnr = 2
                else:
                    telnr = "X"
                typ = code.get(cell, "")
                out.append(sql_insert(date, telnr, INITIALS[hdr], typ))

    return out


def convert_na(lines: list[str], verbose: bool) -> list[str]:
    skip = {"", "X", "x", "L", "SD", "sd", "HQ", "hq", "CPR", "cpr", "MT", "mt", "PD"}
    for i in range(1, 100):
        skip.add(f"L{i}")

    header: list[str] = []
    out: list[str] = []

    for line in lines:
        line = line.strip()
        if not line:
            continue
        if verbose:
            print(f"NA line: {line}", file=sys.stderr, flush=True)
        split = parse_csv_line(line)

        if "DOW,Date" in line:
            header = [h for h in split if h]
            continue

        date = ""
        for key, hdr in enumerate(header):
            if key >= len(split):
                continue
            cell = split[key]
            if hdr == "Date" and not date:
                date = parse_date(cell)
            elif hdr in INITIALS and cell not in skip:
                typ = cell.lower()
                out.append(sql_insert(date, 0, INITIALS[hdr], typ))

    return out


def convert_sa(lines: list[str], verbose: bool) -> list[str]:
    skip = {"", "-"}
    out: list[str] = []

    for line in lines:
        line = line.strip()
        if not line:
            continue
        if verbose:
            print(f"SA line: {line}", file=sys.stderr, flush=True)
        if "Date" in line:
            continue

        split = parse_csv_line(line)
        if not split:
            continue

        # TODO (special case):
        #       if more than one set of initials ("MM/NN" in alias column),
        #       currently creates alias=""
        #       remedy is to duplicate that line and remove "/NN" from original alias, 
        #       and remove the duplicated line's "MM/" portion of the alias
        # Note: asumes only one "/", could there/has there ever been more than two "/"?
        date = parse_date(split[0])
        for i in (1, 2):
            if i >= len(split):
                continue
            cell = split[i]
            if cell in skip:
                continue
            oncall = "oc" if "oc" in cell else ""
            initials = cell.replace("oc", "")
            alias = INITIALS_SA.get(initials, "")
            typ = f"sa{oncall}"
            out.append(sql_insert(date, i, alias, typ))

    return out



In [ ]:

# def convert_eeoc(lines: list[str], verbose: bool) -> list[str]:
#     out: list[str] = []
#     date_key: int | None = None
#     alias_key: int | None = None

#     for line in lines:
#         line = line.strip()
#         if not line:
#             continue
#         if verbose:
#             print(f"EEOC line: {line}", file=sys.stderr, flush=True)

#         split = parse_csv_line(line)
#         if "Date" in split:
#             date_key = split.index("Date")
#             alias_key = split.index("Alias") if "Alias" in split else None
#             continue

#         if date_key is None or alias_key is None:
#             continue
#         if date_key >= len(split) or alias_key >= len(split):
#             continue

#         date_value = split[date_key]
#         first_space = date_value.find(" ")
#         if first_space != -1:
#             date_value = date_value[first_space + 1 :].strip()
#         date = parse_date(date_value)

#         alias = split[alias_key].strip()
#         if alias:
#             out.append(sql_insert(date, 0, alias, "eeoc"))

#     return out

In [ ]:
def main() -> None:

    # --- option 1: arguments for API call (with config file) ---

    # --- option 2: arguments from command line (standalone call) ---
    # args = parse_args()
    # if not args.input.exists():
    #     raise FileNotFoundError(f"Input file does not exist: {args.input}")

    # --- option 3: arguments hardcoded here (run or test) ---
    # Provide type, date range, input filespec, and output filespec here for testing or running in notebook

    verbose = True  # Set verbose to True for debugging output

    # TODO: required date range: 
    # source file may include dates outside this range, but only dates within this range should be processed
    #start_date = datetime.date(2026, 7, 24)
    #end_date = datetime.date(2026, 10, 31)

    # type = 'eeoc'   # electrical engineering on call (default, do not use)
    # type = 'sa'     # staff astronomer
    type = 'oa'     # observing assistant
    # type = 'na'     # night attendant

    if not type:
        print('Please use the --type option with "sa", "oa", or "na".')
        return

    match type:
        case 'sa':
            input_file = Path("../input_files/2026B_SA_Staff_Schedule.csv")
            output_file = Path("../output_files/2026B_SA_Staff_Schedule.sql")
        case 'oa':
            input_file = Path("../input_files/2026B_OA_Staff_Schedule.csv")
            output_file = Path("../output_files/2026B_OA_Staff_Schedule.sql")
        case 'na':
            input_file = Path("../input_files/2026B_NA_Staff_Schedule.csv")
            output_file = Path("../output_files/2026B_NA_Staff_Schedule.sql-2")
        case 'eeoc':
            print('EEOC is not supported. Please use the --type option with "sa", "oa", or "na".')
            return
        case _:
            print(f'Unsupported type: {type}')
            return

    if not input_file.exists():
        raise FileNotFoundError(f"Input file does not exist: {input_file}")

    if not input_file.suffix.lower() == ".csv":
        raise ValueError(f"Input file must be a CSV: {input_file}")

    # ===== end option 3: arguments hardcoded here (run or test) =====

    print(f" Input file: {input_file}")
    print(f"Output file: {output_file}")

    lines = read_lines(input_file)
    for line in lines:
        if line.strip():
            print(f"First non-empty line: {line}")
            break

    converters = {
        "oa": convert_oa,
        "na": convert_na,
        "sa": convert_sa,
        # "eeoc": convert_eeoc,
    }

    # sql_lines = converters[args.type](lines, args.verbose)
    # emit(sql_lines, args.output)

    sql_lines = converters[type](lines, verbose)
    emit(sql_lines, output_file)

    print("\nConversion completed!\n")

In [13]:
main()

OA line: Date,DOW,K1 PI,Institution,K1 Instrument,Date,DOW,CJ,JRK,HH,JP,TR,TC,RM,MW,MP,Mtg,K2 Instrument,Institution,K2 PI last,Date,DOW
OA line: 2026-07-24,FRI,Liu,UH,KPF,2026-07-24,FRI,,K1,L,L,O4,L,O2,K2,K2O,,NIRSPEC,CIT,Hillenbrand,2026-07-24,FRI
OA line: 2026-07-25,SAT,Liu,UH,KPF,2026-07-25,SAT,,K1,X,O7,O5,X,O3,O1,K2,,NIRC2-LGS/NIRC2-NGS,UCSB/UH,Millar-Blanchaer/Williams,2026-07-25,SAT
OA line: 2026-07-26,SUN,Crossfield/Weiss,NASA/Notre Dame,KPF/KPF-CC,2026-07-26,SUN,,K1,X,K1O,O6,O1,O4,O2,K2,,TBD,KECK,Engineering,2026-07-26,SUN
OA line: 2026-07-27,MON,Ghez/Howard,UCLA/CIT,OSIRIS-LGS/KPF-CC,2026-07-27,MON,,O1,L,K1,K2O,O2,O5,O3,K2,,NIRC2-LGS/NIRSPEC,UCLA/Subaru,Ghez/Kawakita,2026-07-27,MON
OA line: 2026-07-28,TUE,Konopacky/Korhonen Cuestas,UCSD/Northwestern,OSIRIS-LGS/MOSFIRE,2026-07-28,TUE,,O2,L,K1,K2,O3,O6,O4,O1,,NIRC2-LGS/NIRSPEC,UCLA/Subaru,Ghez/Kawakita,2026-07-28,TUE
OA line: 2026-07-29,WED,Konopacky/Korhonen Cuestas,UCSD/Northwestern,OSIRIS-LGS/MOSFIRE,2026-07-29,WED,,O3,L,K1,

 Input file: ../input_files/2026B_OA_Staff_Schedule.csv
Output file: ../output_files/2026B_OA_Staff_Schedule.sql
First non-empty line: Date,DOW,K1 PI,Institution,K1 Instrument,Date,DOW,CJ,JRK,HH,JP,TR,TC,RM,MW,MP,Mtg,K2 Instrument,Institution,K2 PI last,Date,DOW
Done!


<!-- if __name__ == "__main__":
    main() -->